# Phase 0-5: CRC-VAL confusion matrix · per-class F1 (eval-only)

**목적:** Acc-3 메인 표 4 arm(lf=1.0, seed=42)의 **best FT checkpoint**로 CRC-VAL을 재평가하고, 혼동행렬·클래스별 F1을 저장한다. **재학습 없음.**

**대상 run (4개):** `macenko_none`, `imagenet_init`, `curriculum`, `none`

---

### Kaggle 실행

1. **Add Input:** `CRC-VAL-HE-7K`, `NCT-CRC-HE-100K-NONORM`(split JSON 탐색용, 선택)
2. **Add Input:** FT best checkpoint Dataset — 아래 4파일:
   - `ft_macenko_none_seed42_lf1.00_best.pt`
   - `ft_imagenet_init_seed42_lf1.00_best.pt`
   - `ft_curriculum_seed42_lf1.00_best.pt`
   - `ft_none_seed42_lf1.00_best.pt`
3. `B_Phase0 확인 재실험` 폴더를 `/kaggle/working/`에 업로드 후 **Run All**

checkpoint는 `02_confusion_eval/A_checkpoints/` 또는 `/kaggle/input/**/` 에 두면 자동 탐색.

**산출:** `02_confusion_eval/B_결과/confusion/*.png`, `01_<tag>_per_class_f1.csv`, `02_confusion_eval_summary.json`

## 0. 경로·설정

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import functional as TF

NCT_CLASS_DIRS = ("ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM")
SEED = 42
LABEL_FRAC = 1.0
ACC_TOLERANCE = 0.002

RUNS = [
    {"tag": "macenko_none", "use_macenko": True, "expected_acc": 0.8451253481894151},
    {"tag": "imagenet_init", "use_macenko": False, "expected_acc": 0.5809192200557103},
    {"tag": "curriculum", "use_macenko": False, "expected_acc": 0.6069637883008356},
    {"tag": "none", "use_macenko": False, "expected_acc": 0.5619777158774373},
]

ON_KAGGLE = Path("/kaggle/working").exists()
CWD = Path.cwd().resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ON_KAGGLE:", ON_KAGGLE, "DEVICE:", DEVICE)

In [ ]:
def find_phase0_root() -> Path:
    markers = [
        "02_confusion_eval/A_notebooks/01_confusion_eval.ipynb",
        "01_split_overlap/A_notebooks/01_split_overlap_check.ipynb",
    ]
    candidates = [CWD, CWD.parent, CWD.parent.parent,
                    Path("/kaggle/working/B_Phase0 확인 재실험")]
    for c in candidates:
        if any((c / m).exists() for m in markers):
            return c if c.name.startswith("B_Phase0") else c.parent
        if c.name.startswith("B_Phase0"):
            return c
    return CWD


PHASE0_ROOT = find_phase0_root()
STEP2_ROOT = PHASE0_ROOT / "02_confusion_eval"
DIR_CKPT_LOCAL = STEP2_ROOT / "A_checkpoints"
DIR_RESULT = STEP2_ROOT / "B_결과"
DIR_CONFUSION = DIR_RESULT / "confusion"
for d in (DIR_CKPT_LOCAL, DIR_RESULT, DIR_CONFUSION):
    d.mkdir(parents=True, exist_ok=True)

print("PHASE0_ROOT:", PHASE0_ROOT)
print("DIR_CKPT_LOCAL:", DIR_CKPT_LOCAL)
print("DIR_RESULT:", DIR_RESULT)

## 1. 데이터·모델 유틸 (Acc-2 정합)

In [ ]:
KAGGLE_CRC_CANDIDATES = [
    Path("/kaggle/input/crc-val-he-7k/CRC-VAL-HE-7K"),
    Path("/kaggle/input/crc-val-he-7k"),
    Path("/kaggle/input/datasets/imrankhan77/crc-val-he-7k/CRC-VAL-HE-7K"),
]


def resolve_class_root(path: Path) -> Optional[Path]:
    if not path.is_dir():
        return None
    hits = sum(1 for c in NCT_CLASS_DIRS if (path / c).is_dir())
    if hits >= 3:
        return path
    for child in sorted(path.iterdir()):
        if child.is_dir() and sum(1 for c in NCT_CLASS_DIRS if (child / c).is_dir()) >= 3:
            return child
    return None


def find_crc_root() -> Path:
    for cand in KAGGLE_CRC_CANDIDATES:
        found = resolve_class_root(cand)
        if found is not None:
            return found
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in sorted(input_root.rglob("*")):
            if p.is_dir() and ("val" in str(p).lower() or "7k" in str(p).lower()):
                found = resolve_class_root(p)
                if found is not None:
                    return found
    raise FileNotFoundError("CRC-VAL-HE-7K not found. Add Input on Kaggle.")


class ImageFolderRGB(Dataset):
    def __init__(self, root: str, size: int, class_subset: Optional[List[str]] = None):
        self.base = datasets.ImageFolder(root)
        if class_subset:
            keep = {i for i, n in enumerate(self.base.classes) if n in class_subset}
            self.indices = [i for i, (_, y) in enumerate(self.base.samples) if y in keep]
            self.remap = {old: new for new, old in enumerate(sorted(keep))}
        else:
            self.indices = list(range(len(self.base)))
            self.remap = {i: i for i in range(len(self.base.classes))}
        self.to_tensor = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor()])

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        path, y = self.base.samples[self.indices[i]]
        from PIL import Image
        return self.to_tensor(Image.open(path).convert("RGB")), self.remap[y]


def try_macenko(x: torch.Tensor) -> torch.Tensor:
    mean = x.mean(dim=(-2, -1), keepdim=True)
    std = x.std(dim=(-2, -1), keepdim=True).clamp_min(1e-4)
    return ((x - mean) / std * 0.2 + 0.5).clamp(0, 1)


class Classifier(nn.Module):
    def __init__(self, num_classes: int, tag: str):
        super().__init__()
        if tag == "imagenet_init":
            try:
                net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            except Exception:
                net = models.resnet18(pretrained=True)
        else:
            try:
                net = models.resnet18(weights=None)
            except TypeError:
                net = models.resnet18(pretrained=False)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
        self.net = net

    def forward(self, x):
        return self.net(x)


CRC_ROOT = find_crc_root()
probe = datasets.ImageFolder(str(CRC_ROOT))
class_names = [c for c in NCT_CLASS_DIRS if c in probe.classes][:9]
crc_full = ImageFolderRGB(str(CRC_ROOT), 224, class_subset=class_names)
crc_ds = Subset(crc_full, list(range(min(7180, len(crc_full)))))
crc_loader = DataLoader(crc_ds, batch_size=64, shuffle=False, num_workers=2)
NUM_CLASSES = len(class_names)
print("CRC_ROOT:", CRC_ROOT)
print("classes:", class_names, "n=", len(crc_ds))

## 2. checkpoint 탐색·평가

In [ ]:
def ckpt_filename(tag: str) -> str:
    return f"ft_{tag}_seed{SEED}_lf{LABEL_FRAC:.2f}_best.pt"


def find_best_ckpt(tag: str) -> Path:
    name = ckpt_filename(tag)
    candidates = [DIR_CKPT_LOCAL / name]
    exp_root = PHASE0_ROOT.parent.parent.parent / "09_보완_재실험"
    candidates.append(exp_root / "outputs" / "checkpoints" / name)
    candidates.append(Path("/kaggle/working/09_보완_재실험/outputs/checkpoints") / name)
    for p in candidates:
        if p.exists():
            return p
    input_root = Path("/kaggle/input")
    if input_root.exists():
        hits = list(input_root.rglob(name))
        if hits:
            return sorted(hits, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    raise FileNotFoundError(f"{name} not found. Put in A_checkpoints/ or Kaggle Dataset.")


@torch.no_grad()
def predict_crc(model, loader, use_macenko: bool) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        if use_macenko:
            x = torch.stack([try_macenko(x[i].cpu()) for i in range(x.size(0))]).to(DEVICE)
        pred = model(x).argmax(1).cpu().numpy()
        preds.append(pred)
        labels.append(y.numpy())
    return np.concatenate(labels), np.concatenate(preds)


def save_confusion_fig(cm: np.ndarray, tag: str, acc: float) -> Path:
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"{tag} seed{SEED} lf{LABEL_FRAC:.2f} | CRC Acc={acc:.4f}")
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_ylabel("true")
    ax.set_xlabel("pred")
    fig.colorbar(im, ax=ax, fraction=0.046)
    out = DIR_CONFUSION / f"01_{tag}_seed{SEED}_lf{LABEL_FRAC:.2f}_confusion.png"
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    return out

## 3. 4 arm 루프 (Run All)

In [ ]:
summary_rows = []

for run in RUNS:
    tag = run["tag"]
    use_macenko = run["use_macenko"]
    expected = run["expected_acc"]
    ckpt_path = find_best_ckpt(tag)
    print("\n===", tag, "===", ckpt_path)

    model = Classifier(NUM_CLASSES, tag).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])

    y_true, y_pred = predict_crc(model, crc_loader, use_macenko)
    acc = float((y_true == y_pred).mean())
    delta = abs(acc - expected)
    ok = delta <= ACC_TOLERANCE
    print(f"  CRC Acc={acc:.6f} expected={expected:.6f} delta={delta:.6f} {'OK' if ok else 'MISMATCH'}")

    report = classification_report(
        y_true, y_pred, target_names=class_names, digits=4, zero_division=0, output_dict=True
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig_path = save_confusion_fig(cm, tag, acc)

    per_class_rows = []
    for cls_name in class_names:
        stats = report.get(cls_name, {})
        per_class_rows.append({
            "tag": tag,
            "class": cls_name,
            "precision": stats.get("precision", np.nan),
            "recall": stats.get("recall", np.nan),
            "f1": stats.get("f1-score", np.nan),
            "support": stats.get("support", np.nan),
        })
    csv_path = DIR_RESULT / f"01_{tag}_seed{SEED}_lf{LABEL_FRAC:.2f}_per_class_f1.csv"
    pd.DataFrame(per_class_rows).to_csv(csv_path, index=False)

    np.savetxt(
        DIR_CONFUSION / f"02_{tag}_seed{SEED}_lf{LABEL_FRAC:.2f}_confusion.csv",
        cm, fmt="%d", delimiter=",",
    )

    summary_rows.append({
        "tag": tag,
        "seed": SEED,
        "label_frac": LABEL_FRAC,
        "use_macenko_eval": use_macenko,
        "ckpt_path": str(ckpt_path),
        "crc_acc": acc,
        "expected_acc": expected,
        "acc_delta": delta,
        "acc_ok": ok,
        "macro_f1": report.get("macro avg", {}).get("f1-score", np.nan),
        "confusion_png": str(fig_path),
        "per_class_csv": str(csv_path),
    })
    print("  saved:", fig_path.name, csv_path.name)

summary_path = DIR_RESULT / "02_confusion_eval_summary.json"
summary_path.write_text(json.dumps(summary_rows, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(summary_rows).to_csv(DIR_RESULT / "03_confusion_eval_summary.csv", index=False)
print("\nWrote:", summary_path)
print(pd.DataFrame(summary_rows)[["tag", "crc_acc", "expected_acc", "acc_ok", "macro_f1"]])